In [ ]:
"""
02_hotspot_identification.py  
------------------------------------------
Usage:
    python 02_hotspot_identification.py <*_wstreetInfos.csv> [OPTIONS]

Options:
    --output-dir    Output directory (default: ./data/outputs/)
    --hospital-lat  Hospital latitude  WGS84 (default: 48.83931, HEGP)
    --hospital-lon  Hospital longitude WGS84 (default:  2.27537, HEGP)
    --min-count     Min occurrences to flag as hotspot (default: 30)
    --exclude-cp    Postal code to exclude from histogram (default: 75015)
"""

In [ ]:
import argparse
import os
import sys
from datetime import datetime

In [ ]:
parser = argparse.ArgumentParser()
parser.add_argument("input_file")
parser.add_argument("--output-dir",   default="./data/outputs/")
parser.add_argument("--hospital-lat", type=float, default=48.83931098867617)
parser.add_argument("--hospital-lon", type=float, default=2.2753718727661623)
parser.add_argument("--min-count",    type=int,   default=30)
parser.add_argument("--exclude-cp",   type=int,   default=75015)
args = parser.parse_args()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.geometry import Point

In [ ]:
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
sys.path.insert(0, BASE_DIR)
from methods.methods import calculer_distance_euclidienne

In [ ]:
os.makedirs(args.output_dir, exist_ok=True)

── Column-name repair guard ──────────────────────────────────────────────────
Defensive utility for an upstream pipeline behaviour that can silently replace
column names with integer indices (e.g. after certain cudf read_csv / concat
operations where the schema is not written to the file).

Usage: call repair_column_names(df, reference_df) right after any read_csv
or concat that exhibits this symptom, before passing the frame downstream.

  df_loaded = pd.read_csv(path, sep=";")
  if df_loaded.columns.dtype == int:          # symptom check
      df_loaded = repair_column_names(df_loaded, reference_df)

In [ ]:
def repair_column_names(df: pd.DataFrame,
                        reference: pd.DataFrame) -> pd.DataFrame:
    """
    Restore string column names when they have been replaced by integer indices.

    Parameters
    ----------
    df        : DataFrame with integer column indices (the broken frame).
    reference : Any DataFrame that has the correct column names in the same
                positional order (e.g. a small head() of the same file read
                with explicit dtype=str, or the source DataFrame before saving).

    Returns
    -------
    DataFrame with columns renamed to match *reference*.
    """
    expected = list(reference.columns)
    if len(df.columns) != len(expected):
        raise ValueError(
            f"Cannot repair: df has {len(df.columns)} columns, "
            f"reference has {len(expected)}."
        )
    return df.rename(columns=dict(zip(df.columns, expected)))
fig_dir = "./figures/"
os.makedirs(fig_dir, exist_ok=True)

In [ ]:
date = datetime.today().strftime("%Y%m%d")
print(f"[02] start — date={date}")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §1  Load — only the columns we actually use
# ═════════════════════════════════════════════════════════════════════════════
print("[02] §1 Loading data...")

In [ ]:
df = pd.read_csv(args.input_file, sep=";", usecols=[
    "adr_init", "ville_init", "cp_init",
    "latitude", "longitude",
    "contains_street_type",
])
df["contains_street_type"] = df["contains_street_type"].astype(bool)

In [ ]:
df_w_street = df[df["contains_street_type"]].copy()
print(f"  Total: {len(df):,}  |  With street type: {len(df_w_street):,}")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §2  Group-count repeated addresses
# ═════════════════════════════════════════════════════════════════════════════
print("[02] §2 Computing group counts...")

In [ ]:
GROUP_KEYS = ["adr_init", "ville_init", "cp_init"]

In [ ]:
df_counts = (
    df_w_street
    .groupby(GROUP_KEYS)
    .agg(
        count=("adr_init", "size"),
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
    )
    .reset_index()
)

In [ ]:
# agg already deduplicates — no need for a separate merge + drop_duplicates
print(f"  Distinct candidate addresses: {len(df_counts):,}")
print(f"  Max count: {df_counts['count'].max()}")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §3  Distribution plots
# ═════════════════════════════════════════════════════════════════════════════
print("[02] §3 Saving plots...")

In [ ]:
df_plot = df_counts[
    (df_counts["cp_init"] != args.exclude_cp) &
    (df_counts["count"]   >= args.min_count)
]

In [ ]:
if len(df_plot) > 0:
    # Linear histogram
    fig, ax = plt.subplots()
    ax.hist(df_plot["count"], bins=50, edgecolor="black")
    ax.set(title="Distribution of grouped address counts",
           xlabel="Count", ylabel="Frequency")
    fig.savefig(os.path.join(fig_dir, f"count_distribution_{date}.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig)

    # Log-log histogram
    bins = np.logspace(
        np.log10(max(df_plot["count"].min(), 1)),
        np.log10(df_plot["count"].max()),
        50,
    )
    fig, ax = plt.subplots()
    ax.hist(df_plot["count"], bins=bins, edgecolor="black")
    ax.set(xscale="log", yscale="log",
           title="Log-Log histogram",
           xlabel="Count (log)", ylabel="Frequency (log)")
    ax.grid(True, which="both", ls="--", linewidth=0.5)
    fig.savefig(os.path.join(fig_dir, f"count_loglog_{date}.png"),
                dpi=150, bbox_inches="tight")
    plt.close(fig)

    print(f"  Plots saved → {fig_dir}")
else:
    print("  No candidates above threshold — plots skipped.")

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §4  GeoDataFrame + distance from hospital
# ═════════════════════════════════════════════════════════════════════════════
print("[02] §4 Computing distances (Lambert-93)...")

In [ ]:
gdf = gpd.GeoDataFrame(
    df_counts,
    geometry=gpd.points_from_xy(df_counts["longitude"], df_counts["latitude"]),
    crs="EPSG:4326",
).to_crs("EPSG:2154")

In [ ]:
# Hospital reference point
hosp = (
    gpd.GeoDataFrame(
        {"geometry": [Point(args.hospital_lon, args.hospital_lat)]},
        crs="EPSG:4326",
    )
    .to_crs("EPSG:2154")
)

In [ ]:
gdf["distance_m"]  = calculer_distance_euclidienne(
    gdf.geometry.x.values,
    gdf.geometry.y.values,
    hosp.geometry.x.values,
    hosp.geometry.y.values,
)
gdf["distance_km"] = gdf["distance_m"] / 1000

In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# §5  Flag hotspots and save
# ═════════════════════════════════════════════════════════════════════════════
print("[02] §5 Flagging hotspots...")

In [ ]:
gdf["is_hotspot"] = gdf["count"] >= args.min_count
print(f"  Hotspot addresses (count ≥ {args.min_count}): {gdf['is_hotspot'].sum():,}")

In [ ]:
# Propagate flag back to full patient-level dataset
df_out = df.merge(
    gdf[GROUP_KEYS + ["count", "distance_km", "is_hotspot"]],
    on=GROUP_KEYS,
    how="left",
)
df_out["is_hotspot"] = df_out["is_hotspot"].fillna(False)

In [ ]:
out_hotspot = os.path.join(args.output_dir, f"patients_geocoded_hotspot_{date}.csv")
df_out.to_csv(out_hotspot, sep=";", index=False)

In [ ]:
out_cand = os.path.join(args.output_dir, f"hotspot_candidates_{date}.csv")
gdf.drop(columns="geometry").to_csv(out_cand, sep=";", index=False)

In [ ]:
print(f"  Hotspot data      → {out_hotspot}")
print(f"  Candidates table  → {out_cand}")
print("[02] DONE.")